# Notebook Test

In [2]:
# import

from IPython.display import Video
import cv2
import easyocr
from paddleocr import PaddleOCR
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans

import pandas as pd
import numpy as np
from tqdm import tqdm

/home/colinhl/anaconda3/envs/sdd2025/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/colinhl/anaconda3/envs/sdd2025/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [3]:
user = "USER_alpedhuez"
video_id = "VIDEO_6814175723704683782"
video_path = f"videos_mp4/downloads/{user}/{video_id}.mp4"
Video(video_path, width=250)

In [4]:
# load dataset

df_train = pd.read_csv("X_train.csv", sep=';')
df_test = pd.read_csv("X_test.csv", sep=';')
df_train.head()

,id,album,artist,artists,aspect_ratio,channel,description,video_duration,format,release_year,track,uploader,filepath,download_timing,uploader_short,vid,uid
0,7602656035161050390,NaN,Urhov Bogdan,['Urhov Bogdan'],0.56,Davos Klosters,you dream you 🥹 #davosklosters #skiing #mounta...,13,1080x1920,2026,оригинальный звук,davosklosters,downloads/USER_davosklosters/VIDEO_76026560351...,2026-02-12 09:29:25,davos,VIDEO_7602656035161050390,USER_davosklosters
1,7590718903144287510,NaN,LykTraffx,['LykTraffx'],0.56,Davos Klosters,already missing this again 🥹 #spenglercup #dav...,12,1080x1920,2026,оригинальный звук,davosklosters,downloads/USER_davosklosters/VIDEO_75907189031...,2026-02-12 09:29:25,davos,VIDEO_7590718903144287510,USER_davosklosters
2,7571821778746592534,NaN,ALTÉGO,['ALTÉGO'],0.56,Davos Klosters,how??!!!🥹🥹 #davosklosters #skiing #ski #season...,13,1080x1920,2025,THE FATE OF OPHELIA X MIDNIGHT CITY,davosklosters,downloads/USER_davosklosters/VIDEO_75718217787...,2026-02-12 09:29:25,davos,VIDEO_7571821778746592534,USER_davosklosters
3,7569927329154190614,NaN,Raye,['Raye'],0.56,Davos Klosters,We're back on snow!🥹🎿 #davosklosters #seasonop...,7,1080x1920,2025,original sound,davosklosters,downloads/USER_davosklosters/VIDEO_75699273291...,2026-02-12 09:29:25,davos,VIDEO_7569927329154190614,USER_davosklosters
4,7566270741134462230,NaN,músicas e traduções,['músicas e traduções'],0.56,Davos Klosters,"Somebody send help, pls. 🥶 #davos #firstsnow #...",8,1080x1920,2025,som original,davosklosters,downloads/USER_davosklosters/VIDEO_75662707411...,2026-02-12 09:29:25,davos,VIDEO_7566270741134462230,USER_davosklosters


# Extraction d'images avec OpenCV

In [3]:
cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Ici, 'frame' est une image (matrice NumPy)
    # Vous pouvez calculer des features ici (ex: luminosité moyenne)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
cap.release()

# Extraction des couleurs principales

In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans

def extract_dominant_colors(video_path, n_clusters=3, sample_rate=30):
    cap = cv2.VideoCapture(video_path)
    pixels = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # On ne prend qu'une frame sur 30 (environ 1 par seconde)
        if int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % sample_rate == 0:
            # On redimensionne pour accélérer le calcul (le clustering est gourmand)
            small_frame = cv2.resize(frame, (56, 100))
            # OpenCV utilise BGR, on repasse en RGB pour une interprétation humaine simple
            rgb_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB) # image RGB de taille (height, width, 3)
            pixels.append(rgb_frame.reshape(-1, 3)) # listes de tous les pixels Red, Green, Blue
            
    cap.release()

    # On regroupe tous les pixels échantillonnés
    all_pixels = np.vstack(pixels)

    # Application de K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=5)
    kmeans.fit(all_pixels)

    # Récupération des couleurs (centres des clusters)
    colors = kmeans.cluster_centers_.astype(int)
    
    # Optionnel : Calculer le pourcentage de chaque couleur
    labels = kmeans.labels_
    counts = np.bincount(labels)
    percentages = counts / len(labels)

    return colors, percentages

# Utilisation
video_id = "VIDEO_6821547365887905030"
video_path = f"videos_mp4/downloads/{user}/{video_id}.mp4"

top_colors, weights = extract_dominant_colors(video_path)

print("Couleurs dominantes (RGB) :")
for i, color in enumerate(top_colors):
    print(f"Couleur {i+1}: {color} - Présence: {weights[i]*100:.1f}%")

Couleurs dominantes (RGB) :
Couleur 1: [116 139 157] - Présence: 31.6%
Couleur 2: [167 187 206] - Présence: 62.6%
Couleur 3: [43 46 39] - Présence: 5.7%


In [15]:
df = df_test

df_colors = pd.DataFrame(columns=["video_id", 'R1', 'G1', 'B1', 'W1', 'R2', 'G2', 'B2', 'W2', 'R3', 'G3', 'B3', 'W3'])

for path, vid in tqdm(zip(df['filepath'], df['vid']), total = len(df), desc="Recherche Couleurs Principales"):
    # calcul des couleurs principales
    top_colors, color_weights = extract_dominant_colors(f"videos_mp4/{path}", n_clusters=3, sample_rate=60)
    
    new_lign = pd.DataFrame([{
        'video_id': vid, 
        'R1': top_colors[0][0],
        'G1': top_colors[0][1],
        'B1': top_colors[0][2],
        'W1' : color_weights[0],
        'R2': top_colors[1][0],
        'G2': top_colors[1][1],
        'B2': top_colors[1][2],
        'W2' : color_weights[1],
        'R3': top_colors[2][0],
        'G3': top_colors[2][1],
        'B3': top_colors[2][2],
        'W3' : color_weights[2]
    }])
    df_colors = pd.concat([df_colors, new_lign], ignore_index=True)


Recherche Couleurs Principales:   0%|          | 0/338 [00:00<?, ?it/s]/tmp/ipykernel_212456/1343376478.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_colors = pd.concat([df_colors, new_lign], ignore_index=True)
Recherche Couleurs Principales: 100%|██████████| 338/338 [09:28<00:00,  1.68s/it]  


In [16]:
df_colors.head()

df_colors.to_csv("top_colors_test.csv", sep=",")

# Détection des textes

In [7]:
# On initialise le moteur une seule fois hors de la fonction pour gagner du temps
# gpu=True si vous avez une carte graphique (ou sur Google Colab)
reader = easyocr.Reader(['fr', 'en'], gpu=False)


def extract_text_from_video(video_path):
    """
    Analyse seulement 3 images clés : 1s après début, milieu, 1s avant fin.
    """
    cap = cv2.VideoCapture(video_path)

    # --- Étape 1 : Calculer les positions (en frames) ---
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # 1 seconde après le début, au milieu, 1 seconde avant la fin
    # On utilise max/min pour éviter les erreurs sur les vidéos trop courtes
    pos_debut = int(min(fps, total_frames * 0.1))
    pos_milieu = int(total_frames / 2)
    pos_fin = int(max(0, total_frames - fps))
    
    target_frames = [pos_debut, pos_milieu, pos_fin]
    
    all_detected_text = []

    # --- Étape 2 : Boucle sur les 3 images cibles ---
    for target in target_frames:
        # On déplace le "curseur" de la vidéo à la frame choisie
        cap.set(cv2.CAP_PROP_POS_FRAMES, target)
        ret, frame = cap.read()
        
        if not ret:
            continue
            
        # Redimensionnement (640px de large)
        h, w = frame.shape[:2]
        new_w = 640
        new_h = int(h * (new_w / w))
        img_resized = cv2.resize(frame, (new_w, new_h))

        # FILTRE 1 : Noir et Blanc (Niveaux de gris)
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

        # 2c. FILTRE 2 : Thresholding d'Otsu
        # On utilise THRESH_BINARY_INV pour avoir le texte en BLANC sur fond NOIR
        # (c'est le format que préfère souvent EasyOCR)
        # S'il rate certains textes, essayez THRESH_BINARY (texte noir sur fond blanc).
        _, img_thresholded = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        
        # OCR (detail=0 pour n'avoir que le texte brut)
        results = reader.readtext(img_thresholded, detail=1) # On garde detail=1 pour la confiance
        
        for res in results:
            text = res[1]
            conf = res[2]
            if conf > 0.4:
                all_detected_text.append(text.lower().strip())
    
    cap.release()

    # --- Étape 3 : Nettoyage ---
    unique_text = sorted(list(set([t for t in all_detected_text if len(t) > 2])))
    full_string = " | ".join(unique_text)
    
    return {
        "has_text": len(unique_text) > 0,
        "text_raw": full_string,
        "nb_textbloc": len(unique_text)
    }

Using CPU. Note: This module is much faster with a GPU.


In [8]:
# test du code

video_id = "VIDEO_7604886753471761686"
video_path = f"videos_mp4/downloads/{user}/{video_id}.mp4"

dict = extract_text_from_video(video_path)

print(f"Result : {dict['has_text']}, {dict['text_raw']}, {dict['nb_textbloc']}")

Result : True, estlent | f@rsure, 2


In [ ]:
df = df_train

df_text_train = pd.DataFrame(columns=["video_id", 'nb_textbloc'])

for path, vid in tqdm(zip(df['filepath'], df['vid']), total = len(df), desc="Recherche Text"):
    # extraction text
    results = extract_text_from_video(f"videos_mp4/{path}")
    
    new_lign = pd.DataFrame([{
        'video_id': vid, 
        'nb_textbloc': results['nb_textbloc']
    }])
    df_text_train = pd.concat([df_text_train, new_lign], ignore_index=True)


df = df_test

for path, vid in tqdm(zip(df['filepath'], df['vid']), total = len(df), desc="Recherche Text"):
    # extraction text
    results = extract_text_from_video(f"videos_mp4/{path}")
    
    new_lign = pd.DataFrame([{
        'video_id': vid, 
        'nb_textbloc': results['nb_textbloc']
    }])
    df_text_test = pd.concat([df_text_test, new_lign], ignore_index=True)


Recherche Text: 100%|██████████| 338/338 [23:30<00:00,  4.17s/it]


In [ ]:
df_text_train.to_csv("text_train.csv", sep=",")

df_text_test.to_csv("text_test.csv", sep=",")

,video_id,nb_words
0,VIDEO_7602656035161050390,1
1,VIDEO_7590718903144287510,3
2,VIDEO_7571821778746592534,2
3,VIDEO_7569927329154190614,0
4,VIDEO_7566270741134462230,11
5,VIDEO_7536170254989315350,2
6,VIDEO_7520640970199698710,0
7,VIDEO_7496799936558796054,0
8,VIDEO_7487960173529582870,1
9,VIDEO_7478712216724802838,1


# Détection des textes (V2)

In [ ]:
# 1. Initialisation de PaddleOCR (L'argument use_angle_cls gère le texte penché)
# 'lang="fr"' permet de mieux détecter les accents
ocr = PaddleOCR(use_textline_orientation=True, lang='fr')

def extract_text_from_video_2(video_path, frame_interval=30):
    cap = cv2.VideoCapture(video_path)
    detected_texts = set() # Utilisation d'un set pour éviter les doublons
    
    frame_count = 0
    
    print(f"--- Analyse de la vidéo : {video_path} ---")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # On ne traite qu'une image toutes les 'frame_interval' (ex: 30 = 1s à 30fps)
        if frame_count % frame_interval == 0:
            # Conversion en gris pour améliorer la vitesse (optionnel)
            # frame_process = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            
            # Exécution de l'OCR sur la frame
            result = ocr.predict(frame)

            if result[0] is not None:
                for line in result[0]:
                    text = line[1][0] # Le texte détecté
                    confidence = line[1][1] # Score de confiance (0 à 1)
                    
                    if confidence > 0.8: # On ne garde que ce qui est sûr
                        if text not in detected_texts:
                            print(f"[Frame {frame_count}] Trouvé : {text} ({confidence:.2%})")
                            detected_texts.add(text)

        frame_count += 1

    cap.release()
    print("--- Analyse terminée ---")
    return list(detected_texts)

ValueError: Unknown argument: use_gpu

In [15]:
# Utilisation du script
# Remplacez 'video.mp4' par le chemin de votre vidéo TikTok


video_id = "VIDEO_7022178037378731270"
video_path = f"videos_mp4/downloads/{user}/{video_id}.mp4"
resultats = extract_text_from_video_2(video_path, frame_interval=20)
print(resultats)
# print(f"Result : {dict['has_text']}, {dict['text_raw']}, {dict['nb_textbloc']}")

--- Analyse de la vidéo : videos_mp4/downloads/USER_alpedhuez/VIDEO_7022178037378731270.mp4 ---


NotImplementedError: (Unimplemented) ConvertPirAttribute2RuntimeAttribute not support [pir::ArrayAttribute<pir::DoubleAttribute>]  (at /paddle/paddle/fluid/framework/new_executor/instruction/onednn/onednn_instruction.cc:116)
